In [2]:
# ==============================================================================
# 🌍 PROYECTO XAI SISMOLOGÍA: CNN 1D + Datos Cubanos + Integrated Gradients
# Equipo 6 - Inteligencia Artificial Avanzada
# ==============================================================================

# 1. INSTALACIÓN DE DEPENDENCIAS (Solo se ejecuta la primera vez)
print("📦 Instalando dependencias...")
!pip install -q torch captum matplotlib scikit-learn pandas numpy
print("✅ Dependencias listas.\n")

# 2. IMPORTACIONES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from captum.attr import IntegratedGradients
import os

# Configuración de reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

# 3. CARGA DE DATOS (Intenta leer los archivos, si no, usa los datos internos)
print("📂 Cargando catálogo de sismos cubanos...")
try:
    df_sismos = pd.read_csv("sismos.csv")
    print("✅ Archivo 'sismos.csv' cargado correctamente.")
except FileNotFoundError:
    print("⚠️ 'sismos.csv' no encontrado. Usando catálogo interno...")
    df_sismos = pd.DataFrame({
        'fecha': ['2023-01-15', '2023-03-10', '2023-05-02', '2023-06-25', '2023-08-30', '2023-10-11'],
        'hora': ['03:12:45', '12:05:20', '18:20:03', '21:10:15', '07:55:30', '14:33:12'],
        'latitud': [19.75, 20.12, 19.8, 20.45, 21.1, 20.0],
        'longitud': [-75.17, -74.56, -76.0, -75.9, -77.3, -76.5],
        'profundidad': [10, 15, 12, 8, 18, 5],
        'magnitud': [3.8, 4.1, 3.5, 4.2, 4.0, 3.9],
        'estacion': ['SAN', 'GTB', 'QMBU', 'SCU', 'STG', 'BAY']
    })

print(f"📊 Catálogo cargado con {len(df_sismos)} eventos sísmicos cubanos.\n")

# 4. GENERACIÓN DE DATASET REALISTA BASADO EN EL CATÁLOGO
print("🔨 Generando dataset realista (1000 señales: 500 eventos + 500 ruido)...")
fs = 100          # 100 Hz
longitud = 500    # 5 segundos

def generar_sismo_realista(magnitud, profundidad):
    t = np.arange(longitud) / fs
    amplitud_base = 10 ** (magnitud - 3)  # Escala logarítmica con la magnitud
    factor_atenuacion = np.exp(-profundidad / 50.0)  # Atenuación por profundidad

    senal = np.random.normal(0, 0.05, longitud) # Ruido de fondo

    # Onda P (alta frecuencia, llega primero)
    inicio_p = np.random.randint(80, 150)
    duracion_p = np.random.randint(50, 100)
    t_p = np.arange(duracion_p) / fs
    envolvente_p = np.exp(-4.0 * t_p) * factor_atenuacion
    onda_p = np.sin(2 * np.pi * 8 * t_p) * envolvente_p
    if inicio_p + duracion_p <= longitud:
        senal[inicio_p:inicio_p+duracion_p] += amplitud_base * 0.3 * onda_p

    # Onda S (baja frecuencia, mayor amplitud, llega después)
    inicio_s = inicio_p + np.random.randint(40, 80)
    duracion_s = np.random.randint(100, 200)
    t_s = np.arange(duracion_s) / fs
    envolvente_s = np.exp(-2.5 * t_s) * factor_atenuacion
    onda_s = np.sin(2 * np.pi * 4 * t_s) * envolvente_s
    if inicio_s + duracion_s <= longitud:
        senal[inicio_s:inicio_s+duracion_s] += amplitud_base * onda_s

    # Normalizar
    maximo = np.max(np.abs(senal))
    return senal / maximo if maximo > 0 else senal

def generar_ruido_realista():
    senal = np.random.normal(0, 0.15, longitud)
    senal += np.random.normal(0, 0.05, longitud) # Ruido cultural
    maximo = np.max(np.abs(senal))
    return senal / maximo if maximo > 0 else senal

X_cubano, y_cubano = [], []

# Generar eventos basados en el catálogo real
for _, sismo in df_sismos.iterrows():
    for _ in range(84):  # ~500 eventos en total
        X_cubano.append(generar_sismo_realista(sismo['magnitud'], sismo['profundidad']))
        y_cubano.append(1)

# Generar ruidos
for _ in range(500):
    X_cubano.append(generar_ruido_realista())
    y_cubano.append(0)

X_cubano = np.array(X_cubano[:1000], dtype=np.float32)
y_cubano = np.array(y_cubano[:1000], dtype=np.int64)
tiempo = np.arange(longitud) / fs

print(f"✅ Dataset generado: {X_cubano.shape} | Ruido: {np.sum(y_cubano==0)} | Evento: {np.sum(y_cubano==1)}\n")

# 5. DEFINICIÓN Y ENTRENAMIENTO DE LA CNN 1D
print("🤖 Configurando y entrenando la CNN 1D...")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {dispositivo}")

class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.capas(x)

# Preparar datos
X_tensor = torch.tensor(X_cubano[:, np.newaxis, :], dtype=torch.float32)
y_tensor = torch.tensor(y_cubano, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_cubano)
cargador = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

modelo = CNN1D().to(dispositivo)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)

for epoca in range(15):
    modelo.train()
    perdida_total = 0.0
    for senales, etiquetas in cargador:
        senales, etiquetas = senales.to(dispositivo), etiquetas.to(dispositivo)
        optimizador.zero_grad()
        perdida = criterio(modelo(senales), etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()
    if (epoca + 1) % 5 == 0:
        print(f"  Época {epoca + 1}/15 - Pérdida: {perdida_total/len(cargador):.4f}")

# 6. EVALUACIÓN DEL MODELO
print("\n📊 Evaluando el modelo...")
modelo.eval()
with torch.no_grad():
    salidas = modelo(X_test.to(dispositivo))
    predicciones = torch.argmax(salidas, dim=1).cpu().numpy()

y_real = y_test.cpu().numpy()
exactitud = (predicciones == y_real).mean()

print(f"\n{'='*50}")
print(f"RESULTADOS CNN (Datos Cubanos)")
print(f"{'='*50}")
print(f"Exactitud (Accuracy): {exactitud:.4f}\n")
print("Informe de clasificación:")
print(classification_report(y_real, predicciones, target_names=["Ruido", "Evento Cubano"], zero_division=0))

# Matriz de confusión
matriz = confusion_matrix(y_real, predicciones)
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=["Ruido", "Evento Cubano"])
disp.plot(cmap="Blues")
plt.title("Matriz de Confusión - CNN 1D (Datos Cubanos)")
plt.show()

# 7. EXPLICABILIDAD CON XAI (INTEGRATED GRADIENTS)
print("\n🔍 Aplicando Integrated Gradients (XAI)...")
candidatos = np.where((y_real == 1) & (predicciones == 1))[0]

if len(candidatos) > 0:
    idx = int(candidatos[0])
    senal_xai = X_test[idx:idx+1].to(dispositivo)
    senal_xai.requires_grad = True

    ig = IntegratedGradients(modelo)
    atribuciones, delta = ig.attribute(senal_xai, baselines=torch.zeros_like(senal_xai), target=1, n_steps=100, return_convergence_delta=True)

    senal_np = senal_xai.detach().cpu().numpy()[0, 0]
    relevancia = atribuciones.detach().cpu().numpy()[0, 0]

    with torch.no_grad():
        probs = torch.softmax(modelo(senal_xai), dim=1).cpu().numpy()[0]

    fig, ejes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ejes[0].plot(tiempo, senal_np, color='blue')
    ejes[0].set_title(f"Señal de Evento Cubano (Probabilidad: {probs[1]:.4f})")
    ejes[0].set_ylabel("Amplitud Normalizada")
    ejes[0].grid(True, alpha=0.3)

    ejes[1].plot(tiempo, relevancia, color='red', alpha=0.7)
    ejes[1].axhline(0, color="black", linewidth=0.8)
    ejes[1].set_title("XAI: Atribuciones de Integrated Gradients")
    ejes[1].set_xlabel("Tiempo (s)")
    ejes[1].set_ylabel("Relevancia")
    ejes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f"✅ Delta de convergencia IG: {delta.item():.6f} (Cercano a 0 indica explicación de alta calidad)")
else:
    print("⚠️ No se encontraron eventos correctamente clasificados para explicar.")

print("\n🎉 ¡PROCESO COMPLETADO EXITOSAMENTE! Tu proyecto está listo.")

📦 Instalando dependencias...
✅ Dependencias listas.



d:\Escula MIGUEL\Tercer anno\Inteligencia Artificial\Proyecto XAI Sismologia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Cargando catálogo de sismos cubanos...
⚠️ 'sismos.csv' no encontrado. Usando catálogo interno...
📊 Catálogo cargado con 6 eventos sísmicos cubanos.

🔨 Generando dataset realista (1000 señales: 500 eventos + 500 ruido)...
✅ Dataset generado: (1000, 500) | Ruido: 496 | Evento: 504

🤖 Configurando y entrenando la CNN 1D...
Usando dispositivo: cpu


AttributeError: module 'torch' has no attribute '_utils'